# Évaluation cross-validation — MultiModal MIL (PORPOISE late-fusion)

Analyse des prédictions multi-seeds (k-fold ou Leave-One-Out) produites par :

```
python src/train_cv.py model=multimodal_porpoise ...
```

Source : `outputs/cv_results/multimodal_porpoise_predictions.csv` (chemin stable écrit par `src/train_cv.py`).

Contenu :
1. Métriques par (seed, fold) — AUC, accuracy
2. Agrégation par seed (moyenne ± std), et version "poolée" (utile en LOO)
3. Courbes ROC moyennes
4. Sauvegarde des scores de prédiction dans un CSV global (toutes architectures)
5. Tests de Wilcoxon : stabilité inter-seeds, séparation des classes, comparaison inter-modèles


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eval.cv_metrics import (
    load_predictions, per_group_metrics, aggregate_metrics, pooled_metrics,
    mean_roc_curve, wilcoxon_seed_stability, wilcoxon_class_separation,
    wilcoxon_compare_models, list_available_models,
)

MODEL_NAME = "multimodal_porpoise"
df = load_predictions(MODEL_NAME)
print(df.shape)
df.head()


## 1. Métriques par (seed, fold)

AUC et accuracy calculées séparément pour chaque fold de chaque seed, sur les splits `val` (fold de cross-validation) et `test` (jeu de test indépendant, si configuré).

In [ ]:
metrics = per_group_metrics(df)
metrics


### Agrégation par seed (moyenne ± std des folds)

In [ ]:
agg = aggregate_metrics(metrics)
agg


### Métriques poolées (toutes prédictions d'une seed concaténées)

Utile notamment pour la Leave-One-Out CV, où chaque fold ne contient qu'un seul échantillon (AUC indéfinie par fold).

In [ ]:
pooled = pooled_metrics(df)
pooled


## 2. Courbes ROC moyennes

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for split, color in [("val", "tab:blue"), ("test", "tab:orange")]:
    sub = df[df["split"] == split]
    if sub.empty:
        continue
    mean_fpr, mean_tpr, std_tpr = mean_roc_curve(sub)
    mean_auc = metrics.loc[metrics["split"] == split, "auc"].mean()
    ax.plot(mean_fpr, mean_tpr, label=f"{split} (AUC={mean_auc:.3f})", color=color)
    ax.fill_between(
        mean_fpr,
        np.clip(mean_tpr - std_tpr, 0, 1),
        np.clip(mean_tpr + std_tpr, 0, 1),
        color=color, alpha=0.2,
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title(f"{MODEL_NAME} — ROC moyenne (toutes seeds/folds)")
ax.legend()
plt.show()


## 3. Sauvegarde des scores de prédiction (CSV global)

Concatène les prédictions de ce modèle dans `notebooks/eval_outputs/cv/all_models_predictions.csv`, utilisé pour la comparaison inter-architectures.

In [ ]:
OUT_DIR = Path("eval_outputs") / "cv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_PREDICTIONS_CSV = OUT_DIR / "all_models_predictions.csv"

if ALL_PREDICTIONS_CSV.exists():
    all_preds = pd.read_csv(ALL_PREDICTIONS_CSV)
    all_preds = all_preds[all_preds["model"] != MODEL_NAME]
    all_preds = pd.concat([all_preds, df], ignore_index=True)
else:
    all_preds = df.copy()

all_preds.to_csv(ALL_PREDICTIONS_CSV, index=False)
print(f"Saved {len(df)} rows for '{MODEL_NAME}' -> {ALL_PREDICTIONS_CSV} (total {len(all_preds)} rows)")


## 4. Tests de Wilcoxon

### 4.1 Stabilité entre seeds (AUC par fold)

Test de Wilcoxon signé comparant, fold par fold, l'AUC de la première et de la dernière seed entraînées. Une p-value faible indique que le choix de la seed influence significativement les performances.

In [ ]:
seeds = sorted(metrics["seed"].unique())
val_metrics = metrics[metrics["split"] == "val"]

if len(seeds) >= 2:
    stat, p, n = wilcoxon_seed_stability(val_metrics, seeds[0], seeds[-1])
    print(f"Wilcoxon (AUC par fold) seed {seeds[0]} vs seed {seeds[-1]} : stat={stat}, p={p}, n_pairs={n}")
    if not np.isnan(p):
        print("=> différence significative entre seeds (p<0.05)" if p < 0.05 else "=> pas de différence significative entre seeds")
else:
    print("Une seule seed disponible — pas de test de stabilité entre seeds.")


### 4.2 Séparation des classes

Test de Wilcoxon rank-sum comparant les probabilités prédites pour les échantillons de label 1 vs label 0. Une p-value faible indique que le modèle sépare significativement les deux classes.

In [ ]:
for split in df["split"].unique():
    stat, p = wilcoxon_class_separation(df, split=split)
    print(f"[{split}] Wilcoxon rank-sum (proba classe 1 vs classe 0) : stat={stat:.4f}, p={p:.4g}")
    print("  => séparation significative (p<0.05)" if p < 0.05 else "  => séparation non significative")


### 4.3 Comparaison inter-architectures (AUC, appariée par seed/fold)

Compare l'AUC de ce modèle à celle des autres architectures déjà évaluées (présentes dans `outputs/cv_results/`), via un test de Wilcoxon signé apparié par (seed, fold). Nécessite la même configuration de CV (méthode, k, seeds) pour les modèles comparés.

In [ ]:
other_models = [m for m in list_available_models() if m != MODEL_NAME]
print(f"Modèles disponibles pour comparaison : {other_models}")

comparison_rows = []
for other in other_models:
    other_df = load_predictions(other)
    other_metrics = per_group_metrics(other_df)
    for split in ["val", "test"]:
        a = metrics[metrics["split"] == split]
        b = other_metrics[other_metrics["split"] == split]
        if a.empty or b.empty:
            continue
        stat, p, n = wilcoxon_compare_models(a, b, metric="auc")
        comparison_rows.append({
            "model_a": MODEL_NAME, "model_b": other, "split": split,
            "auc_a": a["auc"].mean(), "auc_b": b["auc"].mean(),
            "wilcoxon_stat": stat, "p_value": p, "n_pairs": n,
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df
